In [2]:
import sys
print(sys.executable)

/home/salim/anaconda3/bin/python3


In [3]:
%pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 2.0 MB/s  0:00:022.0 MB/s eta 0:00:01:01

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /home/salim/anaconda3/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import psycopg2
print("psycopg2 fonctionne !")

psycopg2 fonctionne !


In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from app.config import connexion

# ⚠️ Connexion utilisée UNIQUEMENT pour des requêtes SELECT dans ce
# notebook — aucune donnée de la base réelle n'est modifiée ici.
conn = connexion()
print("Connexion établie avec succès (lecture seule).")

Connexion établie avec succès (lecture seule).


In [6]:
df_horaires = pd.read_sql("SELECT * FROM horaires;", conn)
print(f"{len(df_horaires)} enregistrements chargés.")
df_horaires.head(10)

710 enregistrements chargés.


/tmp/ipykernel_103264/1865660134.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_horaires = pd.read_sql("SELECT * FROM horaires;", conn)


,id,ligne_ref,sens_direction,periode,course,heure_depart,heure_arrivee,code_bus,jour,statut_qualite
0,1,L1,Départ Zanguera,MATIN,Course 1,05:30:00,06:20:00,B1,Lundi-Vendredi,COMPLET
1,2,L1,Départ Zanguera,MATIN,Course 1,05:45:00,06:40:00,B2,Lundi-Vendredi,COMPLET
2,3,L1,Départ Zanguera,MATIN,Course 1,06:00:00,07:00:00,B3,Lundi-Vendredi,COMPLET
3,4,L1,Départ Zanguera,MATIN,Course 1,06:20:00,07:20:00,B4,Lundi-Vendredi,COMPLET
4,5,L1,Départ Zanguera,MATIN,Course 1,06:40:00,07:40:00,B5,Lundi-Vendredi,COMPLET
5,6,L1,Départ Zanguera,MATIN,Course 1,None,08:00:00,B6,Lundi-Vendredi,ARRIVEE_SEULE
6,7,L1,Départ Zanguera,MATIN,Course 2,07:20:00,08:20:00,B1,Lundi-Vendredi,COMPLET
7,8,L1,Départ Zanguera,MATIN,Course 2,07:40:00,09:00:00,B2,Lundi-Vendredi,COMPLET
8,9,L1,Départ Zanguera,MATIN,Course 2,08:00:00,10:30:00,B3,Lundi-Vendredi,DUREE_SUSPECTE
9,10,L1,Départ Zanguera,MATIN,Course 2,08:20:00,11:30:00,B4,Lundi-Vendredi,DUREE_SUSPECTE


In [7]:
manquants = df_horaires.isnull().sum()
pourcentage_manquant = (manquants / len(df_horaires) * 100).round(1)

resume_manquants = pd.DataFrame({
    "valeurs_manquantes": manquants,
    "pourcentage": pourcentage_manquant
}).sort_values("pourcentage", ascending=False)

print("Valeurs manquantes par colonne :")
resume_manquants

Valeurs manquantes par colonne :


,valeurs_manquantes,pourcentage
heure_arrivee,224,31.5
heure_depart,68,9.6
id,0,0.0
ligne_ref,0,0.0
sens_direction,0,0.0
periode,0,0.0
course,0,0.0
code_bus,0,0.0
jour,0,0.0
statut_qualite,0,0.0


In [8]:
doublons = df_horaires.duplicated().sum()
print(f"Doublons exacts détectés : {doublons} sur {len(df_horaires)} "
      f"({doublons/len(df_horaires)*100:.1f}%)")

Doublons exacts détectés : 0 sur 710 (0.0%)


In [9]:
df_valides = df_horaires.dropna(subset=["heure_depart", "heure_arrivee"])
incoherents = df_valides[df_valides["heure_arrivee"] < df_valides["heure_depart"]]
print(f"Enregistrements incohérents (arrivée avant départ) : {len(incoherents)}")
incoherents[["ligne_ref", "jour", "heure_depart", "heure_arrivee"]]

Enregistrements incohérents (arrivée avant départ) : 0


,ligne_ref,jour,heure_depart,heure_arrivee


In [10]:
# Règle de nettoyage appliquée : on garde les horaires incomplets avec un
# statut de qualité, on écarte uniquement les doublons exacts et les
# incohérences temporelles (arrivée avant départ).
df_nettoye = df_horaires.drop_duplicates()

def statut_qualite(ligne):
    if pd.isna(ligne["heure_depart"]) or pd.isna(ligne["heure_arrivee"]):
        return "incomplet"
    if ligne["heure_arrivee"] < ligne["heure_depart"]:
        return "incoherent"
    return "complet"

df_nettoye["statut_qualite"] = df_nettoye.apply(statut_qualite, axis=1)
df_nettoye = df_nettoye[df_nettoye["statut_qualite"] != "incoherent"]

print("Répartition finale après nettoyage :")
print(df_nettoye["statut_qualite"].value_counts())
print(f"\nTotal conservé : {len(df_nettoye)} / {len(df_horaires)} "
      f"({len(df_nettoye)/len(df_horaires)*100:.1f}%)")

Répartition finale après nettoyage :
statut_qualite
complet      418
incomplet    292
Name: count, dtype: int64

Total conservé : 710 / 710 (100.0%)
